# RxNav in a Box Demo

In [13]:
import pandas as pd
import requests
from pandas import json_normalize


# 1. Define the API Endpoint
ndc = '00002147180'

# url = "http://localhost:4000/REST/ndcproperties.json?id=00002147180"
url = f"http://localhost:4000/REST/ndcproperties.json?id={ndc}"

# 2. Make the Request
response = requests.get(url)

# 3. Check if the request was successful (Status Code 200)
if response.status_code == 200:
    # Convert the JSON response into a Python object (usually a list of dicts)
    json_data = response.json()
    
    # 4. Load into Pandas
    # df = pd.DataFrame(json_data)
    
    # Flattens nested JSON (e.g., separates 'address.street', 'address.city')
    df = pd.json_normalize(json_data)
    
    # Display the first few rows
    display(df.head())
else:
    print(f"Error: Failed to retrieve data. Status code: {response.status_code}")

,ndcPropertyList.ndcProperty
0,"[{'ndcItem': '00002147180', 'ndc9': '0002-1471..."


In [20]:
response = json_data["ndcPropertyList"]["ndcProperty"]

display(response)

[{'ndcItem': '00002147180',
  'ndc9': '0002-1471',
  'ndc10': '0002-1471-80',
  'rxcui': '2601770',
  'splSetIdItem': 'd2d7da5d-ad07-4228-955f-cf7e355c8cc0',
  'packagingList': {'packaging': ['4 SYRINGE in 1 CARTON (0002-1471-80)  / .5 mL in 1 SYRINGE (0002-1471-01)']},
  'propertyConceptList': {'propertyConcept': [{'propName': 'LABELER',
     'propValue': 'Eli Lilly and Company'},
    {'propName': 'LABEL_TYPE', 'propValue': 'HUMAN PRESCRIPTION DRUG'},
    {'propName': 'MARKETING_CATEGORY', 'propValue': 'NDA'},
    {'propName': 'MARKETING_EFFECTIVE_TIME_LOW', 'propValue': '20220513'},
    {'propName': 'MARKETING_STATUS', 'propValue': 'ACTIVE'},
    {'propName': 'NDA', 'propValue': 'NDA215866'}]},
  'source': 'Hybrid'}]

In [40]:
rxcui = response[0]["rxcui"]
ndc10 = response[0]["ndc10"]
marketing = response[0]["propertyConceptList"]["propertyConcept"][4]["propValue"]
labeler =  response[0]["propertyConceptList"]["propertyConcept"][0]["propValue"]

print(f"Labeler: {labeler}")
print(f"rxcui: {rxcui}")
print(f"NDC 10: {ndc10}")
print(f"Marketing Status: {marketing}")

Labeler: Eli Lilly and Company
rxcui: 2601770
NDC 10: 0002-1471-80
Marketing Status: ACTIVE


## Alternate

In [17]:
df = pd.json_normalize(json_data)

In [28]:
import pandas as pd

# 1. Normalize the main list to get the base fields
df = pd.json_normalize(
    json_data['ndcPropertyList']['ndcProperty']
)

# 2. Define a helper to turn the 'Name/Value' list into a real dictionary
# It turns: [{'propName': 'LABELER', 'propValue': 'Eli Lilly'}] 
# Into:     {'LABELER': 'Eli Lilly'}
def parse_concepts(concept_list):
    if isinstance(concept_list, list):
        return {item['propName']: item['propValue'] for item in concept_list}
    return {}

# 3. Apply the helper to the nested column
# Note: The column name comes from the nesting path in the JSON
concept_col = 'propertyConceptList.propertyConcept'
parsed_concepts = df[concept_col].apply(parse_concepts)

# 4. Convert those new dicts into a DataFrame and attach them to your main data
df_concepts = pd.json_normalize(parsed_concepts)
# display(df_concepts)

final_df = pd.concat([df, df_concepts], axis=1)

# Optional: Drop the original messy nested columns to clean up
final_df = final_df.drop(columns=[concept_col, 'packagingList.packaging'])

# Display result
# display(final_df[['ndc10', 'LABELER', 'MARKETING_STATUS', 'NDA']])
display(final_df[['ndcItem', 'ndc10', 'rxcui', 'splSetIdItem', 'LABELER', 'MARKETING_STATUS', 'LABEL_TYPE']])

# display(final_df)

,ndcItem,ndc10,rxcui,splSetIdItem,LABELER,MARKETING_STATUS,LABEL_TYPE
0,00002147180,0002-1471-80,2601770,d2d7da5d-ad07-4228-955f-cf7e355c8cc0,Eli Lilly and Company,ACTIVE,HUMAN PRESCRIPTION DRUG
